# **Final Model Comparison & Export (Real Data)**
**Project:** BizFlow360 — ML Early Warning System for Kenyan MSMEs
**Author:** Edusei Mikel Lisamba (Team Lead & ML Integration)

**What this notebook does:** Loads the unified REAL dataset, evaluates all 4 trained models on the exact same test set, generates the final comparison CSV for Chapter 5 of the report, and exports the winning model (LightGBM) as `best_model.joblib` for deployment.

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import shutil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

base_dir = os.path.abspath('..')
print(f"Base directory: {base_dir}")

Base directory: /home/mikel/BizFlow360/ml_models


# 1. Load, Clean, and Prepare Test Data
**What this cell does:** Loads the unified dataset, applies the exact same cleaning and encoding, and splits it to recreate the exact same 20% test set used during training.

In [3]:
# 1. Load Data
df = pd.read_csv(os.path.join(base_dir, 'data', 'unified_msme_modeling_data.csv'))

# 2. Safety Clean
knbs_placeholders = [-19.11, -19.305, -2.4696, -2.4948, -20.58, -73.5, -74.25, -3.528, -8.82]
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].replace(knbs_placeholders, np.nan)
    df[col] = df[col].fillna(df[col].median())

# 3. Encode
le_county = LabelEncoder()
le_sector = LabelEncoder()
df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

# 4. Define Features
features = [
    'county_encoded', 'sector_encoded', 'male_working_owners', 'female_working_owners',
    'total_monthly_expenses', 'monthly_rent_expense', 'monthly_electricity_expense',
    'monthly_credit_expense', 'monthly_social_responsibility_expense',
    'revenue_last_month', 'normal_monthly_revenue', 'net_income_last_month',
    'stock_value_beginning', 'stock_value_end', 'total_turnover_2015',
    'net_income_margin', 'revenue_change_ratio', 'business_closed',
    'number_closed_establishments', 'revenue_decline', 'zero_or_missing_net_income', 'low_revenue'
]

X = df[features]
y = df['distress_label']

# 5. Split and Scale (Must match training random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Test data prepared.")

✅ Test data prepared.


# 2. Load Models and Evaluate
**What this cell does:** Loads the 4 saved `.joblib` models, runs predictions on the test set, and calculates the 5 core metrics for each.

In [7]:
model_paths = {
    'Logistic Regression': os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'logistic_regression_baseline.joblib'),
    'Random Forest': os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'random_forest.joblib'),
    'LightGBM': os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'lightgbm.joblib'),
    'XGBoost': os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'xgboost.joblib')
}

def get_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
    }

results = {}
for name, path in model_paths.items():
    model = joblib.load(path)
    results[name] = get_metrics(model, X_test_scaled, y_test)

# Create DataFrame
comparison_df = pd.DataFrame(results).T

print("="*60)
print(" FINAL REAL DATA MODEL COMPARISON (For Chapter 5)")
print("="*60)
print(comparison_df)
print("="*60)

 FINAL REAL DATA MODEL COMPARISON (For Chapter 5)
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression    0.6532     0.5736  0.1957    0.2918   0.6331
Random Forest          0.6617     0.5336  0.5844    0.5579   0.6969
LightGBM               0.6639     0.5348  0.6113    0.5705   0.7042
XGBoost                0.6586     0.5288  0.5965    0.5606   0.7030


# 3. Export the Winner and the Comparison Table
**What this cell does:** Saves the comparison table to a CSV file for the final report. It then identifies the model with the highest ROC-AUC, copies it to `best_model.joblib`, and overwrites the old preprocessors so the Streamlit app is ready for the real data.

In [8]:
# 1. Save Comparison CSV to the on_real_data metrics subfolder
metrics_dir = os.path.join(base_dir, 'models', 'metrics', 'on_real_data')
os.makedirs(metrics_dir, exist_ok=True)
comparison_df.to_csv(os.path.join(metrics_dir, 'real_data_model_comparison.csv'))
print("✅ Saved real_data_model_comparison.csv to metrics/on_real_data/")

# 2. Identify Winner
best_model_name = comparison_df['ROC-AUC'].idxmax()
best_roc_auc = comparison_df['ROC-AUC'].max()
print(f"\n🏆 WINNER: {best_model_name} (ROC-AUC: {best_roc_auc})")

# 3. Copy Winner to the branded deployment engine
trained_dir = os.path.join(base_dir, 'models', 'trained', 'on_real_data')
os.makedirs(trained_dir, exist_ok=True)

winning_path = model_paths[best_model_name]
engine_path = os.path.join(trained_dir, 'bizflow_engine_v1.0.joblib')
shutil.copy(winning_path, engine_path)
print(f"✅ Exported {best_model_name} to trained/on_real_data/bizflow_engine_v1.0.joblib for deployment.")

# 4. Save the updated preprocessors (fitted on the REAL data)
preproc_dir = os.path.join(base_dir, 'models', 'preprocessing', 'on_real_data')
os.makedirs(preproc_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(preproc_dir, 'scaler.joblib'))
joblib.dump(le_county, os.path.join(preproc_dir, 'le_county.joblib'))
joblib.dump(le_sector, os.path.join(preproc_dir, 'le_sector.joblib'))
print("✅ Updated preprocessors saved to preprocessing/on_real_data/.")

✅ Saved real_data_model_comparison.csv to metrics/on_real_data/

🏆 WINNER: LightGBM (ROC-AUC: 0.7042)
✅ Exported LightGBM to trained/on_real_data/bizflow_engine_v1.0.joblib for deployment.
✅ Updated preprocessors saved to preprocessing/on_real_data/.
